# Visualización de Métricas XAI

Distribuciones (KDE + media) y comparativas por modelo recomendador y algoritmo XAI para **AggDiv**, **IXD**, **ECS** y **MIL**.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from scipy.stats import gaussian_kde
from pathlib import Path
from collections import defaultdict

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.grid':        True,
    'grid.color':       '#ebebeb',
    'grid.linewidth':   0.8,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.family':      'DejaVu Sans',
    'font.size':        11,
})


In [ ]:
MODO = 'semi'   # 'muestra' | 'semi' | 'completo'
MODELOS = None  # None = detectar todos; o lista: ['FunkSVD', 'ItemKNN']
INCLUIR_LEGACY = False  # True solo si quieres leer output/metricas_evaluacion_<modo>

CWD = Path().resolve()
PROJECT_ROOT = next((p for p in [CWD, *CWD.parents] if (p / 'src').exists() and (p / 'output').exists()), CWD.parent.parent)
OUTPUT_ROOT = PROJECT_ROOT / 'output'

METRICAS = ['AggDiv', 'IXD', 'ECS', 'MIL']

# Paleta: azules para kg_, rojos/naranjas para cf_
COLORES_KG = ['#003f9e', '#0077ff', '#00c2c7', '#00875a', '#5b8def', '#00a6a6']
COLORES_CF = ['#cc0000', '#ff6600', '#e6007e', '#ffcc00', '#a23e48', '#f28e2b']


def descubrir_dirs_evaluacion(modo, modelos=None, incluir_legacy=False):
    dirs = []
    if modelos:
        for modelo in modelos:
            eval_dir = OUTPUT_ROOT / modelo / f'metricas_evaluacion_{modo}'
            dirs.append((modelo, eval_dir))
    else:
        for carpeta in sorted(OUTPUT_ROOT.iterdir()) if OUTPUT_ROOT.exists() else []:
            eval_dir = carpeta / f'metricas_evaluacion_{modo}'
            if eval_dir.exists():
                dirs.append((carpeta.name, eval_dir))

    if incluir_legacy:
        legacy_dir = OUTPUT_ROOT / f'metricas_evaluacion_{modo}'
        if legacy_dir.exists():
            dirs.append(('base', legacy_dir))
    return dirs


EVAL_DIRS = descubrir_dirs_evaluacion(MODO, MODELOS, INCLUIR_LEGACY)
OUTPUT_DIR = OUTPUT_ROOT / f'visualizacion_{MODO}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Directorios de evaluaci?n:')
if not EVAL_DIRS:
    print(f'  Sin directorios para modo={MODO}. Ejecuta antes src/evaluacion/pipeline.py --modo {MODO}')
else:
    for modelo, eval_dir in EVAL_DIRS:
        print(f'  {modelo}: {eval_dir}')


In [ ]:
# ?? Carga de datos ????????????????????????????????????????????????????????????
datos: dict = defaultdict(dict)
INFO_ALG = {}


def inferir_metrica(nombre):
    return next((m for m in METRICAS if f'_{m}_' in nombre or nombre.endswith(f'_{m}')), None)


def etiqueta_algoritmo(modelo, algoritmo):
    return f'{modelo} | {algoritmo}'


def etiqueta_legible(clave):
    info = INFO_ALG.get(clave, {})
    modelo = info.get('modelo', '')
    algoritmo = info.get('algoritmo', clave)
    algoritmo = algoritmo.replace('kg_', 'KG: ').replace('cf_', 'CF: ')
    return f'{modelo} | {algoritmo}' if modelo else algoritmo


for modelo_dir_nombre, eval_dir in EVAL_DIRS:
    for csv_path in sorted(eval_dir.glob('evaluacion_*.csv')):
        stem = csv_path.stem
        metrica = inferir_metrica(stem)
        if metrica is None:
            continue

        df = pd.read_csv(csv_path)
        modelo = (
            df['modelo_recomendador'].dropna().iloc[0]
            if 'modelo_recomendador' in df.columns and df['modelo_recomendador'].notna().any()
            else modelo_dir_nombre
        )
        algoritmo = (
            df['algoritmo'].dropna().iloc[0]
            if 'algoritmo' in df.columns and df['algoritmo'].notna().any()
            else stem[len('evaluacion_') : stem.index(f'_{metrica}_')]
        )
        clave = etiqueta_algoritmo(modelo, algoritmo)
        df['modelo_recomendador'] = modelo
        df['algoritmo'] = algoritmo
        datos[metrica][clave] = df
        INFO_ALG[clave] = {'modelo': modelo, 'algoritmo': algoritmo}
        print(f'  {metrica:8s}  {clave}')

# Paleta de colores
claves = sorted({a for algs in datos.values() for a in algs})
idx_kg = idx_cf = 0
COLORES = {}
for clave in claves:
    algoritmo = INFO_ALG.get(clave, {}).get('algoritmo', clave)
    if algoritmo.startswith('kg_'):
        COLORES[clave] = COLORES_KG[idx_kg % len(COLORES_KG)]; idx_kg += 1
    else:
        COLORES[clave] = COLORES_CF[idx_cf % len(COLORES_CF)]; idx_cf += 1

print(f'\nM?tricas: {list(datos.keys())}')
print(f'Modelos: {sorted({v["modelo"] for v in INFO_ALG.values()})}')


In [ ]:
# ── Función KDE: una figura por recomendador, leyenda fuera del plot ─────────
def _modelos_en_metrica(metrica):
    """Devuelve los modelos recomendadores únicos con datos para esa métrica."""
    return sorted({INFO_ALG[a]['modelo'] for a in datos.get(metrica, {})})


def plot_kde(metrica, columna, ylabel, titulo_base, guardar_prefix=None):
    """Genera UNA figura por cada modelo recomendador.
    La leyenda se coloca siempre a la derecha del área de la gráfica, sin taparla.
    """
    if metrica not in datos:
        print(f'Sin datos: {metrica}')
        return

    modelos = _modelos_en_metrica(metrica)
    if not modelos:
        print(f'Sin modelos para {metrica}')
        return

    for modelo in modelos:
        algs_modelo = sorted(
            a for a in datos[metrica]
            if INFO_ALG[a]['modelo'] == modelo
        )

        handles = []
        # Figura más ancha para que el plot no quede apretado con la leyenda fuera
        fig, ax = plt.subplots(figsize=(13, 5))

        for alg in algs_modelo:
            df = datos[metrica][alg]
            if columna not in df.columns:
                continue
            vals = df[columna].dropna().values
            if len(vals) < 2 or len(np.unique(vals)) < 2:
                continue
            color = COLORES.get(alg, '#888')
            algoritmo_label = INFO_ALG[alg]['algoritmo'].replace('kg_', 'KG: ').replace('cf_', 'CF: ')
            mu = np.mean(vals)

            x = np.linspace(vals.min(), vals.max(), 400)
            kde_f = gaussian_kde(vals, bw_method='scott')
            ax.plot(x, kde_f(x), color=color, linewidth=2)
            ax.axvline(mu, color=color, linewidth=1.2, linestyle='--', alpha=0.8)

            handles.append(mlines.Line2D([], [], color=color, linewidth=2,
                                         label=f'{algoritmo_label}  (μ={mu:.3f}, n={len(vals)})'))

        if not handles:
            plt.close(fig)
            continue

        ax.set_xlabel(columna, fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(f'{titulo_base} — {modelo}', fontsize=12, fontweight='bold', pad=10)

        # Leyenda fuera a la derecha — bbox_to_anchor en coordenadas de axes
        legend = ax.legend(
            handles=handles,
            fontsize=9,
            framealpha=0.9,
            loc='center left',
            bbox_to_anchor=(1.02, 0.5),
            borderaxespad=0,
        )

        # Reservar margen derecho fijo para la leyenda; savefig usa bbox_inches=tight
        fig.subplots_adjust(right=0.68)

        if guardar_prefix:
            nombre = OUTPUT_DIR / f'{guardar_prefix}_{modelo}.png'
            fig.savefig(nombre, dpi=150, bbox_inches='tight')
            print(f'  💾 {nombre.name}')

        plt.show()
        plt.close(fig)


## AggDiv — Diversidad Agregada
Número de hoteles explicadores únicos por usuario. Mayor = más diverso.


In [ ]:
if 'AggDiv' in datos:
    ejemplo = next(iter(datos['AggDiv'].values()))
    cols = [c for c in ejemplo.columns if c.startswith('AggDiv')]
    for col in cols:
        plot_kde(
            metrica        = 'AggDiv',
            columna        = col,
            ylabel         = 'densidad',
            titulo_base    = f'AggDiv — distribución de {col}',
            guardar_prefix = OUTPUT_DIR / f'kde_AggDiv_{col}',
        )


## IXD — Inter-eXplanation Diversity
Diversidad entre explicadores de distintas recomendaciones del mismo usuario. Rango [0, 1].


In [ ]:
if 'IXD' in datos:
    ejemplo = next(iter(datos['IXD'].values()))
    cols = [c for c in ejemplo.columns if c.startswith('IXD')]
    for col in cols:
        plot_kde(
            metrica        = 'IXD',
            columna        = col,
            ylabel         = 'densidad',
            titulo_base    = f'IXD — distribución de {col}',
            guardar_prefix = OUTPUT_DIR / f'kde_IXD_{col}',
        )


## ECS — Explanation Consistency Score
Solapamiento Jaccard de explicadores entre usuarios que recibieron el mismo hotel. Rango [0, 1].


In [ ]:
# Resumen ECS
if 'ECS' in datos:
    filas = []
    for alg, df in datos['ECS'].items():
        fila = {'algoritmo': alg}
        for col in [c for c in df.columns if c.startswith('ECS')]:
            fila[col] = round(df[col].dropna().mean(), 4)
        fila['n_hoteles'] = len(df)
        filas.append(fila)
    df_ecs = pd.DataFrame(filas).set_index('algoritmo')
    print(df_ecs.to_string())


In [ ]:
# KDE de la distribución de ECS por hotel
if 'ECS' in datos:
    ejemplo = next(iter(datos['ECS'].values()))
    cols = [c for c in ejemplo.columns if c.startswith('ECS')]
    for col in cols:
        plot_kde(
            metrica        = 'ECS',
            columna        = col,
            ylabel         = 'densidad',
            titulo_base    = f'ECS — distribución de {col} por hotel recomendado',
            guardar_prefix = OUTPUT_DIR / f'kde_ECS_{col}',
        )


In [ ]:
# Scatter ECS vs popularidad — una figura por recomendador
if 'ECS' in datos:
    modelos_ecs = sorted({INFO_ALG[a]['modelo'] for a in datos['ECS']})
    for modelo in modelos_ecs:
        algs_modelo = sorted(a for a in datos['ECS'] if INFO_ALG[a]['modelo'] == modelo)
        fig, ax = plt.subplots(figsize=(13, 5))
        handles = []
        for alg in algs_modelo:
            df = datos['ECS'][alg]
            if 'ECS' not in df.columns or 'n_usuarios' not in df.columns:
                continue
            sub = df[['n_usuarios', 'ECS']].dropna()
            if sub.empty:
                continue
            color = COLORES.get(alg, '#888')
            algoritmo_label = INFO_ALG[alg]['algoritmo'].replace('kg_', 'KG: ').replace('cf_', 'CF: ')
            ax.scatter(sub['n_usuarios'], sub['ECS'], color=color, alpha=0.4, s=16, edgecolors='none')
            if len(sub) > 2:
                z = np.polyfit(np.log1p(sub['n_usuarios']), sub['ECS'], 1)
                xs = np.sort(sub['n_usuarios'].values)
                ax.plot(xs, np.poly1d(z)(np.log1p(xs)), color=color, linewidth=1.8, alpha=0.9)
            handles.append(mlines.Line2D([], [], color=color, linewidth=2, label=algoritmo_label))

        if not handles:
            plt.close(fig)
            continue

        ax.set_xlabel('Nº usuarios que recibieron el hotel (popularidad)', fontsize=11)
        ax.set_ylabel('ECS  (Jaccard medio)', fontsize=11)
        ax.set_title(f'ECS vs Popularidad — {modelo}', fontsize=12, fontweight='bold', pad=10)
        ax.set_ylim(-0.02, 1.02)

        ax.legend(
            handles=handles,
            fontsize=9,
            framealpha=0.9,
            loc='center left',
            bbox_to_anchor=(1.02, 0.5),
            borderaxespad=0,
        )
        fig.subplots_adjust(right=0.68)
        fig.savefig(OUTPUT_DIR / f'scatter_ECS_vs_popularidad_{modelo}.png', dpi=150, bbox_inches='tight')
        plt.show()
        plt.close(fig)


## MIL — Mean Inter-List Diversity
Métrica de sistema: un único valor por algoritmo. Rango [0, 1].


In [ ]:
if 'MIL' in datos:
    filas = []
    for alg, df in datos['MIL'].items():
        fila = {'algoritmo': alg}
        for col in [c for c in df.columns if c.startswith('MIL')]:
            fila[col] = df[col].iloc[0]
        filas.append(fila)
    df_mil = pd.DataFrame(filas).set_index('algoritmo')
    print(df_mil.to_string())

    modelos_mil = sorted({INFO_ALG[a]['modelo'] for a in df_mil.index})

    for col in df_mil.columns:
        for modelo in modelos_mil:
            algs_modelo = [a for a in df_mil.index if INFO_ALG[a]['modelo'] == modelo]
            if not algs_modelo:
                continue

            vals = [df_mil.loc[a, col] for a in algs_modelo]
            labels = [
                INFO_ALG[a]['algoritmo'].replace('kg_', 'KG: ').replace('cf_', 'CF: ')
                for a in algs_modelo
            ]
            cols_bar = [COLORES.get(a, '#888') for a in algs_modelo]

            fig, ax = plt.subplots(figsize=(8, max(3, len(algs_modelo) * 0.65)))
            bars = ax.barh(labels, vals, color=cols_bar, alpha=0.85, height=0.45)
            ax.bar_label(bars, fmt='%.4f', padding=4, fontsize=9)
            ax.set_xlabel(f'{col}  (0 = sin personalización, 1 = totalmente personalizado)', fontsize=10)
            ax.set_title(f'MIL — {col} · {modelo}', fontsize=12, fontweight='bold', pad=10)
            ax.set_xlim(0, 1.18)
            ax.invert_yaxis()
            plt.tight_layout()
            nombre = OUTPUT_DIR / f'barras_MIL_{col}_{modelo}.png'
            fig.savefig(nombre, dpi=150, bbox_inches='tight')
            print(f'  💾 {nombre.name}')
            plt.show()
            plt.close(fig)


## Panel resumen — las 4 métricas de un vistazo


In [ ]:
# Panel resumen por recomendador — las 4 métricas de un vistazo
modelos_panel = sorted({v['modelo'] for v in INFO_ALG.values()})

for modelo in modelos_panel:
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(
        f'Métricas XAI — {modelo}',
        fontsize=13, fontweight='bold', y=1.02
    )

    def algs_de(metrica):
        return sorted(a for a in datos.get(metrica, {}) if INFO_ALG[a]['modelo'] == modelo)

    # AggDiv y IXD: KDE
    for ax, (metrica, col) in zip(axes[:2], [('AggDiv', 'AggDiv'), ('IXD', 'IXD')]):
        if metrica not in datos:
            ax.set_title(f'{metrica}: sin datos'); continue
        handles = []
        for alg in algs_de(metrica):
            df = datos[metrica][alg]
            if col not in df.columns: continue
            vals = df[col].dropna().values
            if len(vals) < 2 or len(np.unique(vals)) < 2: continue
            color = COLORES.get(alg, '#888')
            lbl = INFO_ALG[alg]['algoritmo'].replace('kg_', 'KG: ').replace('cf_', 'CF: ')
            x = np.linspace(vals.min(), vals.max(), 300)
            ax.plot(x, gaussian_kde(vals)(x), color=color, linewidth=2)
            ax.axvline(np.mean(vals), color=color, linewidth=1, linestyle='--', alpha=0.7)
            handles.append(mlines.Line2D([], [], color=color, linewidth=2, label=lbl))
        ax.set_title(metrica, fontsize=11, fontweight='bold')
        ax.set_xlabel(col, fontsize=9)
        ax.set_ylabel('densidad', fontsize=9)
        if handles:
            ax.legend(handles=handles, fontsize=6, framealpha=0.9,
                      loc='upper right', bbox_to_anchor=(1, 1))

    # ECS: KDE
    ax = axes[2]
    handles = []
    for alg in algs_de('ECS'):
        df = datos['ECS'][alg]
        if 'ECS' not in df.columns: continue
        vals = df['ECS'].dropna().values
        if len(vals) < 2 or len(np.unique(vals)) < 2: continue
        color = COLORES.get(alg, '#888')
        lbl = INFO_ALG[alg]['algoritmo'].replace('kg_', 'KG: ').replace('cf_', 'CF: ')
        x = np.linspace(0, 1, 300)
        ax.plot(x, gaussian_kde(vals)(x), color=color, linewidth=2)
        ax.axvline(np.mean(vals), color=color, linewidth=1, linestyle='--', alpha=0.7)
        handles.append(mlines.Line2D([], [], color=color, linewidth=2, label=lbl))
    ax.set_title('ECS', fontsize=11, fontweight='bold')
    ax.set_xlabel('ECS', fontsize=9)
    ax.set_ylabel('densidad', fontsize=9)
    ax.set_xlim(0, 1)
    if handles:
        ax.legend(handles=handles, fontsize=6, framealpha=0.9,
                  loc='upper right', bbox_to_anchor=(1, 1))
    elif 'ECS' not in datos:
        ax.set_title('ECS: sin datos')

    # MIL: barras horizontales
    ax = axes[3]
    algs_mil = algs_de('MIL')
    if algs_mil:
        mil_vals_m = {a: datos['MIL'][a]['MIL'].iloc[0]
                      for a in algs_mil if 'MIL' in datos['MIL'][a].columns}
        labels_mil = [INFO_ALG[a]['algoritmo'].replace('kg_', 'KG: ').replace('cf_', 'CF: ')
                      for a in mil_vals_m]
        bars = ax.barh(labels_mil, list(mil_vals_m.values()),
                       color=[COLORES.get(a, '#888') for a in mil_vals_m], alpha=0.85, height=0.45)
        ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=7)
        ax.set_xlim(0, 1.15)
        ax.invert_yaxis()
        ax.set_title('MIL', fontsize=11, fontweight='bold')
        ax.set_xlabel('MIL', fontsize=9)
    else:
        ax.set_title('MIL: sin datos')

    plt.tight_layout()
    nombre = OUTPUT_DIR / f'panel_metricas_{modelo}.png'
    fig.savefig(nombre, dpi=150, bbox_inches='tight')
    print(f'💾 {nombre.name}')
    plt.show()
    plt.close(fig)
